# CRSP Sector ETF Price Extraction — Data Collection

**Academic research only — not investment advice.**

This is the market-data counterpart to `01_ravenpack_news_extraction.ipynb`, built in the same silver/gold shape. It pulls daily price/return/volume history for the eleven SPDR sector ETFs (the index-level proxies used throughout this project, not individual stocks) from `crsp.stksecurityinfohist` (security/ticker history) joined to `crsp.dsf_v2` (CRSP CIZ daily stock file) — the schema validated in `Basic_EDA_Analysis.ipynb`.

Two outputs:

1. **Bronze/Silver — raw daily security pull** (`data_collection/raw/crsp_sector_etf_daily_raw_<START>_<END>.csv`): one row per ETF per CRSP trading session, straight off the join, no engineered features.
**Overnight gap (added 2026-08-07):** the pull now also takes `d.dlyopen`, and the gold panel carries `overnight_gap` (prev close → open) and `open_to_close` (open → close). Pre-open news is absorbed in the gap, which is *not* capturable by trading at the open — separating the two is what lets the modelling notebooks distinguish a research finding about predictability from a tradeable one.

2. **Gold — `market_daily_df`** (`data_collection/market_daily_df.csv`): cleaned, deduplicated, mapped to sector asset names, with 1-day and 5-day forward return/label features engineered for the baseline vs. sentiment-augmented classifier comparison.

**Licensing note:** CRSP is also a licensed WRDS dataset. As a precaution, the row-level pull is written only to `data_collection/raw/`, which is `.gitignore`'d — same conservative treatment as the RavenPack silver table. Only the engineered gold table is intended to be committed.

In [1]:
%pip install -q wrds psycopg2-binary pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wrds

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 120

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT
    NOTEBOOK_DIR = REPO_ROOT / "data_collection"
elif PROJECT_ROOT.name == "data_collection" and (PROJECT_ROOT.parent / "data_collection").is_dir():
    REPO_ROOT = PROJECT_ROOT.parent
    NOTEBOOK_DIR = PROJECT_ROOT
else:
    raise FileNotFoundError("Run this notebook from the repository root or data_collection/.")
RAW_DIR = NOTEBOOK_DIR / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_DATE = "2015-01-01"
END_DATE = "2026-12-31"

SECTOR_ETF_TO_ASSET = {
    "XLK": "Technology",
    "XLV": "Health_Care",
    "XLF": "Financials",
    "XLC": "Communication_Services",
    "XLY": "Consumer_Discretionary",
    "XLI": "Industrials",
    "XLP": "Consumer_Staples",
    "XLE": "Energy",
    "XLU": "Utilities",
    "XLB": "Materials",
    "XLRE": "Real_Estate",
}
TARGET_TICKERS = tuple(SECTOR_ETF_TO_ASSET.keys())
FORWARD_HORIZONS = [1, 5]

RAW_PRICES_CSV = RAW_DIR / f"crsp_sector_etf_daily_raw_{START_DATE[:4]}_{END_DATE[:4]}.csv"
MARKET_DAILY_CSV = NOTEBOOK_DIR / "market_daily_df.csv"

print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Sector ETF universe: {TARGET_TICKERS}")
print(f"Bronze/Silver output (gitignored, raw/): {RAW_PRICES_CSV}")
print(f"Gold output (committed, engineered):     {MARKET_DAILY_CSV}")

Date range: 2020-01-01 to 2026-12-31
Sector ETF universe: ('XLK', 'XLV', 'XLF', 'XLC', 'XLY', 'XLI', 'XLP', 'XLE', 'XLU', 'XLB', 'XLRE')
Bronze/Silver output (gitignored, raw/): C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\crsp_sector_etf_daily_raw_2020_2026.csv
Gold output (committed, engineered):     C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\market_daily_df.csv


In [3]:
# Connect to WRDS. This may prompt for credentials if no .pgpass file is configured.
db = wrds.Connection()

crsp_tables = db.list_tables(library="crsp")
required_crsp_tables = ["stksecurityinfohist", "dsf_v2"]
missing_crsp_tables = [table for table in required_crsp_tables if table not in crsp_tables]

if missing_crsp_tables:
    raise RuntimeError(f"Missing CRSP tables: {missing_crsp_tables}")

print("WRDS connection ready.")
print("Required CRSP tables found:", required_crsp_tables)

Loading library list...


Done
WRDS connection ready.
Required CRSP tables found: ['stksecurityinfohist', 'dsf_v2']


## 1. Pull raw CRSP daily price/return/volume for the sector ETFs (Bronze/Silver)

`crsp.stksecurityinfohist` gives the ticker-to-permno mapping with valid date ranges; `crsp.dsf_v2` (CRSP CIZ) has the daily return/price/volume. This is the schema that was validated against the broad-market ETF proxies in `Basic_EDA_Analysis.ipynb` ("CRSP CIZ crsp.dsf_v2"), now pointed at the eleven SPDR sector ETFs.

In [4]:
def sql_string_list(values):
    """Return a SQL-safe single-quoted literal list for simple identifier strings."""
    return ", ".join("'" + str(value).replace("'", "''") + "'" for value in values)


ticker_sql = sql_string_list(TARGET_TICKERS)

market_query = f"""
    WITH names AS (
        SELECT DISTINCT
            permno,
            ticker,
            secinfostartdt AS name_start,
            COALESCE(secinfoenddt, DATE '9999-12-31') AS name_end
        FROM crsp.stksecurityinfohist
        WHERE ticker IN ({ticker_sql})
          AND secinfostartdt <= DATE '{END_DATE}'
          AND COALESCE(secinfoenddt, DATE '9999-12-31') >= DATE '{START_DATE}'
    )
    SELECT
        d.dlycaldt AS session_date,
        n.ticker,
        d.permno,
        d.dlyret AS daily_return,
        d.dlyprc AS price,
        d.dlyopen AS open_price,
        d.dlyvol AS volume
    FROM crsp.dsf_v2 d
    INNER JOIN names n
        ON d.permno = n.permno
       AND d.dlycaldt BETWEEN n.name_start AND n.name_end
    WHERE d.dlycaldt BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
    ORDER BY n.ticker, d.dlycaldt
"""

market_daily_raw = db.raw_sql(market_query)

tickers_found = set(market_daily_raw["ticker"].dropna().unique())
missing_tickers = set(TARGET_TICKERS) - tickers_found
if missing_tickers:
    raise RuntimeError(f"Could not retrieve all sector ETF tickers from CRSP: missing {missing_tickers}")

print(f"Raw CRSP rows pulled: {len(market_daily_raw):,}")
print(f"Tickers found: {sorted(tickers_found)}")
display(market_daily_raw.head())

Raw CRSP rows pulled: 16,588
Tickers found: ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']


,session_date,ticker,permno,daily_return,price,open_price,volume
0,2020-01-02,XLB,86449,-0.011723,60.7,61.83,7357433.0
1,2020-01-03,XLB,86449,-0.016145,59.72,60.08,12423416.0
2,2020-01-06,XLB,86449,-0.004354,59.46,59.55,15764364.0
3,2020-01-07,XLB,86449,-0.001177,59.39,59.36,20267054.0
4,2020-01-08,XLB,86449,0.003536,59.6,59.4,8079600.0


In [5]:
# Licensed WRDS data — write only to the gitignored raw/ folder, never to a tracked path.
market_daily_raw.to_csv(RAW_PRICES_CSV, index=False)
print(f"Saved bronze/silver (raw CRSP pull) table to: {RAW_PRICES_CSV}")
print("This path is under data_collection/raw/, which is .gitignore'd - do not force-add it.")

Saved bronze/silver (raw CRSP pull) table to: C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\crsp_sector_etf_daily_raw_2020_2026.csv
This path is under data_collection/raw/, which is .gitignore'd - do not force-add it.


## 2. Clean & engineer the daily sector-asset panel (Gold): `market_daily_df`

Dedup, map ticker → sector asset name, and engineer 1-day and 5-day forward return/label features — same logic as the ETF proxy panel in `Basic_EDA_Analysis.ipynb`, applied to the eleven SPDR sector ETFs.

In [6]:
market_daily_df = market_daily_raw.copy()
market_daily_df["session_date"] = pd.to_datetime(market_daily_df["session_date"])
market_daily_df["daily_return"] = pd.to_numeric(market_daily_df["daily_return"], errors="coerce")
market_daily_df["price"] = pd.to_numeric(market_daily_df["price"], errors="coerce").abs()
# CRSP encodes a bid/ask average as a negative price; .abs() is applied to open for the same reason.
market_daily_df["open_price"] = pd.to_numeric(market_daily_df["open_price"], errors="coerce").abs()
market_daily_df["volume"] = pd.to_numeric(market_daily_df["volume"], errors="coerce")
market_daily_df["asset"] = market_daily_df["ticker"].map(SECTOR_ETF_TO_ASSET)

market_daily_df = (
    market_daily_df
    .dropna(subset=["asset", "session_date"])
    .sort_values(["asset", "session_date", "permno"])
)
duplicate_asset_sessions = market_daily_df.duplicated(["asset", "session_date"]).sum()
if duplicate_asset_sessions:
    raise ValueError(f"CRSP extraction has {duplicate_asset_sessions:,} duplicate asset/session rows; resolve security-history overlap before continuing.")
market_daily_df = market_daily_df.reset_index(drop=True)

# Decompose the session into the part a trader cannot capture (the overnight gap, which is where
# pre-open news is absorbed) and the part they can (open -> close). These are raw price ratios rather
# than distribution-adjusted returns, so they will not compound exactly to `daily_return` on
# ex-dividend dates; the discrepancy is a few basis points and is checked in section 3.
market_daily_df["prev_close"] = market_daily_df.groupby("asset")["price"].shift(1)
market_daily_df["overnight_gap"] = market_daily_df["open_price"] / market_daily_df["prev_close"] - 1
market_daily_df["open_to_close"] = market_daily_df["price"] / market_daily_df["open_price"] - 1
market_daily_df["gap_positive"] = np.where(
    market_daily_df["overnight_gap"].notna(),
    (market_daily_df["overnight_gap"] > 0).astype(float),
    np.nan,
)
market_daily_df["open_to_close_positive"] = np.where(
    market_daily_df["open_to_close"].notna(),
    (market_daily_df["open_to_close"] > 0).astype(float),
    np.nan,
)

for horizon in FORWARD_HORIZONS:
    shifted_product = pd.Series(1.0, index=market_daily_df.index)
    for lag in range(1, horizon + 1):
        shifted_product = shifted_product * (1 + market_daily_df.groupby("asset")["daily_return"].shift(-lag))
    fwd_col = f"fwd_{horizon}d_return"
    label_col = f"fwd_{horizon}d_positive"
    market_daily_df[fwd_col] = shifted_product - 1
    market_daily_df[label_col] = np.where(
        market_daily_df[fwd_col].notna(),
        (market_daily_df[fwd_col] > 0).astype(float),
        np.nan,
    )

print(f"Gold panel shape: {market_daily_df.shape}")
print(market_daily_df.groupby(["asset", "ticker"]).agg(
    start=("session_date", "min"),
    end=("session_date", "max"),
    sessions=("session_date", "nunique"),
    nonnull_returns=("daily_return", "count"),
))
display(market_daily_df.head())

Gold panel shape: (16588, 17)
                                   start        end  sessions  nonnull_returns
asset                  ticker                                                 
Communication_Services XLC    2020-01-02 2025-12-31      1508             1508
Consumer_Discretionary XLY    2020-01-02 2025-12-31      1508             1508
Consumer_Staples       XLP    2020-01-02 2025-12-31      1508             1508
Energy                 XLE    2020-01-02 2025-12-31      1508             1508
Financials             XLF    2020-01-02 2025-12-31      1508             1508
Health_Care            XLV    2020-01-02 2025-12-31      1508             1508
Industrials            XLI    2020-01-02 2025-12-31      1508             1508
Materials              XLB    2020-01-02 2025-12-31      1508             1508
Real_Estate            XLRE   2020-01-02 2025-12-31      1508             1508
Technology             XLK    2020-01-02 2025-12-31      1508             1508
Utilities             

,session_date,ticker,permno,daily_return,price,open_price,volume,asset,prev_close,overnight_gap,open_to_close,gap_positive,open_to_close_positive,fwd_1d_return,fwd_1d_positive,fwd_5d_return,fwd_5d_positive
0,2020-01-02,XLC,17940,0.011747,54.26,53.97,6365149.0,Communication_Services,<NA>,<NA>,0.005373,NaN,1.0,-0.00645,0.0,0.020642,1.0
1,2020-01-03,XLC,17940,-0.00645,53.91,53.67,2351385.0,Communication_Services,54.26,-0.010874,0.004472,0.0,1.0,0.013356,1.0,0.023186,1.0
2,2020-01-06,XLC,17940,0.013356,54.63,53.61,2535880.0,Communication_Services,53.91,-0.005565,0.019026,0.0,1.0,0.001281,1.0,0.019219,1.0
3,2020-01-07,XLC,17940,0.001281,54.7,54.69,2642767.0,Communication_Services,54.63,0.001098,0.000183,1.0,1.0,0.00713,1.0,0.015356,1.0
4,2020-01-08,XLC,17940,0.00713,55.09,54.73,4161180.0,Communication_Services,54.7,0.000548,0.006578,1.0,1.0,0.005264,1.0,0.013069,1.0


## 3. Validation checks

In [7]:
expected_assets = set(SECTOR_ETF_TO_ASSET.values())
actual_assets = set(market_daily_df["asset"].dropna().unique())
missing_assets = expected_assets - actual_assets
assert not missing_assets, f"Missing sector assets: {missing_assets}"

duplicate_rows = market_daily_df.duplicated(["asset", "session_date"]).sum()
assert duplicate_rows == 0, f"Duplicate asset-session rows found: {duplicate_rows}"

# Forward return spot check: today's fwd_1d_return must equal tomorrow's daily_return by asset.
check_df = market_daily_df.sort_values(["asset", "session_date"]).copy()
check_df["next_daily_return"] = check_df.groupby("asset")["daily_return"].shift(-1)
valid_check = check_df[["fwd_1d_return", "next_daily_return"]].dropna()
assert np.allclose(valid_check["fwd_1d_return"], valid_check["next_daily_return"], atol=1e-12), (
    "fwd_1d_return is not shifted correctly"
)

# The overnight gap and the open->close move must compound back to the raw close-to-close price
# change (exactly, up to dividend adjustment which `daily_return` includes and raw prices do not).
recon = market_daily_df.dropna(subset=["overnight_gap", "open_to_close", "prev_close"]).copy()
recon["compounded"] = (1 + recon["overnight_gap"]) * (1 + recon["open_to_close"]) - 1
recon["raw_price_move"] = recon["price"] / recon["prev_close"] - 1
max_gap_error = (recon["compounded"] - recon["raw_price_move"]).abs().max()
assert max_gap_error < 1e-9, f"gap/open-to-close decomposition does not reconcile: {max_gap_error}"

open_missing = market_daily_df["open_price"].isna().mean()
print(f"open_price missing: {open_missing:.2%}")
assert open_missing < 0.02, "more than 2% of sessions are missing an open price"

# How much of the session's move happens before anyone can trade on the news?
share_in_gap = (recon["overnight_gap"].abs().mean()
                / (recon["overnight_gap"].abs().mean() + recon["open_to_close"].abs().mean()))
print(f"share of the average absolute session move that occurs in the overnight gap: {share_in_gap:.1%}")

print("Validation checks passed.")
coverage_summary = market_daily_df.groupby("asset").agg(
    sessions=("session_date", "nunique"),
    start=("session_date", "min"),
    end=("session_date", "max"),
    missing_fwd_1d=("fwd_1d_return", lambda s: s.isna().sum()),
    missing_fwd_5d=("fwd_5d_return", lambda s: s.isna().sum()),
)
display(coverage_summary)

open_price missing: 0.00%
share of the average absolute session move that occurs in the overnight gap: 41.8%
Validation checks passed.


,sessions,start,end,missing_fwd_1d,missing_fwd_5d
asset,,,,,
Communication_Services,1508,2020-01-02,2025-12-31,1.0,5.0
Consumer_Discretionary,1508,2020-01-02,2025-12-31,1.0,5.0
Consumer_Staples,1508,2020-01-02,2025-12-31,1.0,5.0
Energy,1508,2020-01-02,2025-12-31,1.0,5.0
Financials,1508,2020-01-02,2025-12-31,1.0,5.0
Health_Care,1508,2020-01-02,2025-12-31,1.0,5.0
Industrials,1508,2020-01-02,2025-12-31,1.0,5.0
Materials,1508,2020-01-02,2025-12-31,1.0,5.0
Real_Estate,1508,2020-01-02,2025-12-31,1.0,5.0


## 4. Save the gold panel and print a final run summary

`market_daily_df.csv` is engineered (deduplicated, asset-mapped, forward-return features) rather than a raw WRDS export, so it is intended to be committed alongside `news_daily_df.csv` from the RavenPack notebook.

In [8]:
market_daily_df.to_csv(MARKET_DAILY_CSV, index=False)

print("Extraction complete.")
print(f"Bronze/Silver (raw CRSP pull, gitignored): {RAW_PRICES_CSV}  [{len(market_daily_raw):,} rows]")
print(f"Gold (sector panel, committed):            {MARKET_DAILY_CSV}  [{len(market_daily_df):,} rows]")
print(f"Sector ETFs covered: {sorted(market_daily_df['asset'].unique())}")

Extraction complete.
Bronze/Silver (raw CRSP pull, gitignored): C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\raw\crsp_sector_etf_daily_raw_2020_2026.csv  [16,588 rows]
Gold (sector panel, committed):            C:\Users\dongx\OneDrive\文档\Group\siads-699-sentiment-analysis-sp500\data_collection\market_daily_df.csv  [16,588 rows]
Sector ETFs covered: ['Communication_Services', 'Consumer_Discretionary', 'Consumer_Staples', 'Energy', 'Financials', 'Health_Care', 'Industrials', 'Materials', 'Real_Estate', 'Technology', 'Utilities']
